# 🤖 Agente de IA con LangChain + Mistral AI sobre Dataset de Ventas

**Materia:** Inteligencia Artificial — Ingeniería Informática  
**Objetivo:** Construir un agente conversacional que analice un dataset CSV de ventas usando LangChain y el modelo de lenguaje Mistral.

---

### 📋 Flujo general del proyecto

```
Dataset CSV
    │
    ▼
Pandas DataFrame (normalizado)
    │
    ▼
LangChain Pandas Agent
    │
    ▼
Mistral AI (LLM) ◄──► Herramientas Python
    │
    ▼
Respuesta en lenguaje natural
```

> **Dataset fuente:** [Sample Sales Data — Kaggle](https://www.kaggle.com/datasets/kyanyoga/sample-sales-data)  
> **Archivo esperado:** `sales_data.csv` (subir a Colab o montar desde Drive)

---
## 📦 SECCIÓN 1 — Instalación y Configuración de Dependencias

In [48]:
# ============================================================
# CELDA 1: Instalación de librerías necesarias
# ------------------------------------------------------------
# - langchain            : Framework principal del agente
# - langchain-mistralai  : Conector oficial de Mistral para LangChain
# - langchain-experimental: Contiene create_pandas_dataframe_agent
# - pandas               : Manipulación del dataset CSV
# - tabulate             : Formateo de tablas en la salida del agente
# ============================================================

!pip install -q -U \
    langchain \
    langchain-mistralai \
    langchain-experimental \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    pandas \
    tabulate \
    faiss-cpu

print("✅ Instalación completada.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.3 MB/s eta 0:00:00
✅ Instalación completada.


In [49]:
# ============================================================
# CELDA 2: Configuración segura de la API Key de Mistral
# ------------------------------------------------------------
# Se usa google.colab.userdata para leer el secreto guardado
# en Colab bajo el nombre 'MISTRAL_API_KEY'.
#
# ¿Cómo guardar tu API Key en Colab?
#   1. Panel izquierdo → ícono de llave 🔑 (Secrets)
#   2. Crear secreto con nombre: MISTRAL_API_KEY
#   3. Pegar tu clave del portal: https://console.mistral.ai
# ============================================================

import os
from google.colab import userdata

# Leer el secreto y asignarlo como variable de entorno estándar
os.environ["MISTRAL_API_KEY"] = userdata.get("Llavekey")

# Verificación (no imprime la clave completa por seguridad)
api_key = os.environ.get("MISTRAL_API_KEY", "")
if api_key:
    print(f"✅ API Key cargada correctamente: {api_key[:6]}{'*' * (len(api_key) - 6)}")
else:
    raise ValueError("❌ API Key no encontrada. Verifica que el secreto 'MISTRAL_API_KEY' esté configurado en Colab.")

✅ API Key cargada correctamente: PVdHcO**************************


---
## 📂 SECCIÓN 2 — Carga y Normalización del Dataset

In [50]:
# ============================================================
# CELDA 3: Importaciones generales del proyecto
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("✅ Librerías importadas.")

✅ Librerías importadas.


In [51]:
# ============================================================
# CELDA 4: Carga del CSV y función de normalización completa
# ------------------------------------------------------------
# El archivo 'sales_data.csv' puede:
#   - Subirse manualmente: Panel izquierdo → 📁 → Subir archivo
#   - Montarse desde Drive con: drive.mount('/content/drive')
#
# El dataset original de Kaggle usa encoding latin-1 (ISO-8859-1)
# debido a caracteres especiales en nombres de países/ciudades.
# ============================================================


def cargar_y_normalizar_dataset(ruta_csv: str) -> pd.DataFrame:
    """
    Carga el CSV de ventas y aplica un pipeline completo de normalización.

    Pasos:
        1. Carga con encoding correcto
        2. Diagnóstico inicial (info + head)
        3. Normalización de nombres de columnas
        4. Conversión de tipos numéricos
        5. Conversión de fechas
        6. Limpieza de strings
        7. Manejo de nulos

    Parámetros:
        ruta_csv (str): Ruta relativa o absoluta al archivo CSV.

    Retorna:
        pd.DataFrame: DataFrame limpio y normalizado.
    """

    # ----------------------------------------------------------
    # PASO 1: Carga del archivo
    # ----------------------------------------------------------
    print("=" * 60)
    print("📥 PASO 1: Cargando dataset...")
    print("=" * 60)

    df = pd.read_csv(ruta_csv, encoding="latin-1")
    print(f"   Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

    # ----------------------------------------------------------
    # PASO 2: Diagnóstico inicial
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("🔍 PASO 2: Diagnóstico inicial")
    print("=" * 60)
    print("\n--- df.info() ---")
    df.info()
    print("\n--- df.head(3) ---")
    print(df.head(3).to_string())

    # ----------------------------------------------------------
    # PASO 3: Normalización de nombres de columnas
    # strip() elimina espacios y upper() estandariza a mayúsculas
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("🏷️  PASO 3: Normalizando nombres de columnas")
    print("=" * 60)

    df.columns = [col.strip().upper() for col in df.columns]
    print(f"   Columnas: {list(df.columns)}")

    # ----------------------------------------------------------
    # PASO 4: Conversión de columnas numéricas
    # pd.to_numeric con errors='coerce' convierte valores inválidos
    # a NaN en lugar de lanzar un error.
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("🔢 PASO 4: Convirtiendo columnas numéricas")
    print("=" * 60)

    columnas_numericas = {
        "QUANTITYORDERED": "int",
        "PRICEEACH": "float",
        "ORDERLINENUMBER": "int",
        "SALES": "float",
        "MSRP": "float",
    }

    for col, tipo in columnas_numericas.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if tipo == "int":
                # Rellenar NaN antes de convertir a int
                df[col] = df[col].fillna(0).astype(int)
            print(f"   ✔ {col} → {df[col].dtype}")
        else:
            print(f"   ⚠ Columna '{col}' no encontrada, se omite.")

    # ----------------------------------------------------------
    # PASO 5: Conversión de la columna de fechas (ORDERDATE)
    # El formato del dataset es MM/DD/YYYY HH:MM
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("📅 PASO 5: Convirtiendo columna de fechas (ORDERDATE)")
    print("=" * 60)

    if "ORDERDATE" in df.columns:
        df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"], format="mixed", dayfirst=False)
        # Columnas derivadas útiles para consultas
        df["YEAR"]  = df["ORDERDATE"].dt.year
        df["MONTH"] = df["ORDERDATE"].dt.month
        df["MONTH_NAME"] = df["ORDERDATE"].dt.strftime("%B")  # ej. "January"
        print(f"   ✔ ORDERDATE → {df['ORDERDATE'].dtype}")
        print(f"   ✔ Columnas derivadas creadas: YEAR, MONTH, MONTH_NAME")
        print(f"   Rango de fechas: {df['ORDERDATE'].min()} → {df['ORDERDATE'].max()}")

    # ----------------------------------------------------------
    # PASO 6: Limpieza de strings (columnas tipo object)
    # .str.strip() elimina espacios al inicio y al final
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("🧹 PASO 6: Limpiando columnas de texto")
    print("=" * 60)

    columnas_texto = df.select_dtypes(include=["object"]).columns.tolist()
    for col in columnas_texto:
        df[col] = df[col].str.strip()
    print(f"   ✔ Columnas de texto limpiadas: {columnas_texto}")

    # ----------------------------------------------------------
    # PASO 7: Manejo de valores nulos en columnas críticas
    # ADDRESSLINE2 y POSTALCODE son opcionales en muchos registros
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("🔧 PASO 7: Manejando valores nulos")
    print("=" * 60)

    # Columnas opcionales → rellenar con texto descriptivo
    columnas_opcionales = ["ADDRESSLINE2", "STATE", "POSTALCODE"]
    for col in columnas_opcionales:
        if col in df.columns:
            nulos = df[col].isna().sum()
            df[col] = df[col].fillna("N/A")
            print(f"   ✔ {col}: {nulos} nulos → reemplazados con 'N/A'")

    # Columnas críticas → eliminar filas sin valor
    columnas_criticas = ["ORDERNUMBER", "SALES", "CUSTOMERNAME"]
    filas_antes = len(df)
    for col in columnas_criticas:
        if col in df.columns:
            df = df.dropna(subset=[col])
    filas_despues = len(df)
    print(f"   ✔ Filas eliminadas por nulos críticos: {filas_antes - filas_despues}")

    # ----------------------------------------------------------
    # Reporte final
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("✅ NORMALIZACIÓN COMPLETADA")
    print("=" * 60)
    print(f"   Filas finales : {df.shape[0]}")
    print(f"   Columnas      : {df.shape[1]}")
    print(f"   Nulos restantes:\n{df.isnull().sum()[df.isnull().sum() > 0].to_string()}")

    return df


# ------ Ejecutar la carga y normalización ------
RUTA_CSV = "sales_data.csv"   # Ajusta si el archivo está en otra ruta

df = cargar_y_normalizar_dataset(RUTA_CSV)

📥 PASO 1: Cargando dataset...
   Filas: 2823 | Columnas: 25

🔍 PASO 2: Diagnóstico inicial

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   int64  
 1   QUANTITYORDERED   2823 non-null   int64  
 2   PRICEEACH         2823 non-null   float64
 3   ORDERLINENUMBER   2823 non-null   int64  
 4   SALES             2823 non-null   float64
 5   ORDERDATE         2823 non-null   str    
 6   STATUS            2823 non-null   str    
 7   QTR_ID            2823 non-null   int64  
 8   MONTH_ID          2823 non-null   int64  
 9   YEAR_ID           2823 non-null   int64  
 10  PRODUCTLINE       2823 non-null   str    
 11  MSRP              2823 non-null   int64  
 12  PRODUCTCODE       2823 non-null   str    
 13  CUSTOMERNAME      2823 non-null   str    
 14  PHONE             2823 non-null   str

In [52]:
# Rellenar los nulos restantes en la columna TERRITORY
df['TERRITORY'] = df['TERRITORY'].fillna('N/A')
print(f"Nulos restantes en TERRITORY: {df['TERRITORY'].isnull().sum()}")

Nulos restantes en TERRITORY: 0


In [53]:
# ============================================================
# CELDA 5: Vista previa del DataFrame normalizado
# ============================================================

print("📊 Primeras filas del dataset normalizado:")
print(df.head(5).to_string())

print(f"\n📐 Tipos de datos finales:")
print(df.dtypes)

📊 Primeras filas del dataset normalizado:
   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  PRODUCTLINE  MSRP PRODUCTCODE              CUSTOMERNAME             PHONE                   ADDRESSLINE1 ADDRESSLINE2           CITY STATE POSTALCODE COUNTRY TERRITORY CONTACTLASTNAME CONTACTFIRSTNAME DEALSIZE  YEAR  MONTH MONTH_NAME
0        10107               30      95.70                2  2871.00 2003-02-24  Shipped       1         2     2003  Motorcycles    95    S10_1678         Land of Toys Inc.        2125557818        897 Long Airport Avenue          N/A            NYC    NY      10022     USA       N/A              Yu             Kwai    Small  2003      2   February
1        10121               34      81.35                5  2765.90 2003-05-07  Shipped       2         5     2003  Motorcycles    95    S10_1678        Reims Collectables        26.47.1555             59 rue de l'Abbaye          N/A          Reims   N/A 

---
## 🧠 SECCIÓN 3 — Configuración del Modelo Mistral y Creación del Agente

In [54]:
# ============================================================
# CELDA 6: Configuración del modelo Mistral (LLM)
# ------------------------------------------------------------
# Usamos ChatMistralAI con los siguientes parámetros:
#
# - model       : 'mistral-large-latest' — el más capaz de Mistral
#                 (alternativa: 'codestral-latest' para tareas de código)
# - temperature : 0 — sin aleatoriedad, respuestas deterministas
#                 Crucial para consultas numéricas precisas.
# - api_key     : se toma de la variable de entorno configurada antes
# ============================================================

from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,           # Respuestas precisas y reproducibles
    api_key=os.environ["MISTRAL_API_KEY"]
)

print("✅ Modelo Mistral configurado:")
print(f"   Modelo     : {llm.model}")
print(f"   Temperature: {llm.temperature}")

✅ Modelo Mistral configurado:
   Modelo     : mistral-large-latest
   Temperature: 0.0


In [55]:
# ============================================================
# CELDA 7: Creación del Agente de Pandas con LangChain
# ------------------------------------------------------------
# create_pandas_dataframe_agent crea un agente que:
#   1. Recibe una pregunta en lenguaje natural
#   2. Genera código Python/Pandas internamente
#   3. Ejecuta ese código sobre el DataFrame
#   4. Interpreta el resultado y responde en lenguaje natural
#
# Parámetros importantes:
# - verbose=True         : Muestra el razonamiento interno (Thought/Action)
# - allow_dangerous_code : Requerido para permitir ejecución de código
# - agent_type           : OPENAI_FUNCTIONS es el más estable con Mistral
# - handle_parsing_errors: Evita crashes ante errores de parseo del LLM
# - max_iterations       : Límite de pasos para evitar bucles infinitos
# - prefix               : Prompt de sistema personalizado en español
# ============================================================

from langchain_experimental.agents import create_pandas_dataframe_agent

# Prompt de sistema: instruye al agente sobre su rol y comportamiento
SYSTEM_PROMPT = """
Eres un analista de datos experto y estrictamente honesto que trabaja con un dataset de ventas.
Tienes acceso ÚNICAMENTE a un DataFrame de pandas llamado `df`.

## TU ÚNICA FUENTE DE VERDAD
- Solo puedes responder basándote en los datos reales que obtengas ejecutando código sobre `df`.
- NUNCA inventes cifras, nombres, fechas o tendencias.
- NUNCA uses conocimiento externo para completar o "mejorar" una respuesta.
- Si ejecutas código y el resultado está vacío o es None, repórtalo tal cual.

## CUANDO NO TENGAS LA INFORMACIÓN
Si la pregunta requiere datos que no existen en `df`, responde EXACTAMENTE así:
"⚠️ No puedo responder esto con los datos disponibles. El dataset no contiene
información sobre [tema específico]. Las columnas disponibles son: [lista las columnas de df]."

Ejemplos de cuándo aplicar esto:
- Preguntan por un año que no existe en el dataset → di qué años SÍ existen.
- Preguntan por una columna que no existe → di qué columnas SÍ existen.
- Preguntan por proyecciones o predicciones futuras → no las inventes.
- Preguntan por información externa (noticias, precios de mercado, etc.) → fuera de tu alcance.

## ANTES DE RESPONDER, SIEMPRE:
1. Ejecuta código para verificar que los datos existen en `df`.
2. Verifica que el resultado no esté vacío antes de presentarlo.
3. Si el resultado es inesperado o parece incorrecto, dilo explícitamente.

## FORMATO DE RESPUESTA
- Responde SIEMPRE en español.
- Muestra el razonamiento BREVEMENTE antes del código.
- Valores monetarios: símbolo $ con 2 decimales (ej: $12,345.67).
- Tablas: formato legible con columnas alineadas.
- Al final de cada respuesta numérica, indica: "Fuente: columna [X] del dataset."

## SESGOS QUE DEBES EVITAR
- No asumas tendencias que el código no confirme explícitamente.
- No compares con "promedios de la industria" ni datos externos.
- No uses frases como "probablemente", "se estima" o "es posible que" para datos
  que puedes verificar directamente con código.
- Si hay empate en un ranking, repórtalo como empate, no elijas uno arbitrariamente.

## TONO
Eres preciso, directo y transparente. Prefieres decir "no sé" o "no tengo esos datos"
antes que dar una respuesta inventada o incompleta.
"""

agente = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,                     # Ver el proceso Thought → Action → Observation
    agent_type="openai-tools",
    allow_dangerous_code=True,         # Necesario para ejecutar código Python
    handle_parsing_errors=True,        # Tolerancia a errores de formato del LLM
    max_iterations=10,                 # Máximo 10 pasos por consulta
    prefix=SYSTEM_PROMPT
)

print("✅ Agente de LangChain creado exitosamente.")
print(f"   DataFrame shape: {df.shape}")

✅ Agente de LangChain creado exitosamente.
   DataFrame shape: (2823, 28)


---
## 💬 SECCIÓN 4 — Función de Interacción y Casos de Uso

La función `preguntar()` encapsula la invocación al agente y maneja errores de forma amigable.

In [56]:
# ============================================================
# CELDA 8: Función principal para interactuar con el agente
# ============================================================

def preguntar(pregunta: str, mostrar_separador: bool = True) -> str:
    """
    Envía una pregunta en lenguaje natural al agente y retorna la respuesta.

    El agente sigue el ciclo ReAct:
        Thought  → Razona qué hacer
        Action   → Llama una herramienta (Python REPL)
        Observation → Ve el resultado
        Final Answer → Formula la respuesta final

    Parámetros:
        pregunta (str)         : Pregunta en lenguaje natural.
        mostrar_separador (bool): Imprime líneas divisoras para legibilidad.

    Retorna:
        str: Respuesta del agente.
    """
    if mostrar_separador:
        print("\n" + "═" * 60)
        print(f"🧑 PREGUNTA: {pregunta}")
        print("═" * 60)

    try:
        resultado = agente.invoke({"input": pregunta})
        respuesta = resultado.get("output", "Sin respuesta")

        if mostrar_separador:
            print("\n" + "─" * 60)
            print(f"🤖 RESPUESTA FINAL:\n{respuesta}")
            print("─" * 60)

        return respuesta

    except Exception as e:
        error_msg = f"❌ Error al procesar la pregunta: {str(e)}"
        print(error_msg)
        return error_msg


print("✅ Función 'preguntar()' lista para usar.")

✅ Función 'preguntar()' lista para usar.


---
### 🧪 CASO A — Comprensión de Texto y Contexto Descriptivo

Se introduce un párrafo de contexto junto con una pregunta de comprensión.

In [57]:
# ============================================================
# CELDA 9: CASO A — Comprensión de texto
# ------------------------------------------------------------
# Se le proporciona al agente un párrafo descriptivo como
# contexto externo y se le pide que extraiga la idea principal.
#
# Esto demuestra que el agente puede razonar más allá del CSV.
# ============================================================

contexto = """
La empresa Classic Model Cars inició operaciones en 1998 especializándose en la
distribución mayorista de réplicas a escala de vehículos clásicos. Durante los
primeros cinco años, la empresa consolidó una red de clientes en más de 19 países,
priorizando mercados en Europa y América del Norte. Su estrategia comercial se
basa en ofrecer alta calidad a precios competitivos, lo que le permitió superar
a sus principales competidores en ventas de la categoría 'Classic Cars'. En 2004,
la empresa alcanzó su máximo histórico de ingresos, impulsado por el lanzamiento
de nuevas líneas de productos como motocicletas y camiones a escala.
"""

pregunta_a = f"""
Lee el siguiente texto y responde:
1. ¿Cuál es la idea principal del párrafo?
2. ¿En qué año fue el mejor desempeño y qué lo impulsó?
3. ¿Cuántos países mencionan?

Texto:
{contexto}
"""

respuesta_a = preguntar(pregunta_a)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: 
Lee el siguiente texto y responde:
1. ¿Cuál es la idea principal del párrafo?
2. ¿En qué año fue el mejor desempeño y qué lo impulsó?
3. ¿Cuántos países mencionan?

Texto:

La empresa Classic Model Cars inició operaciones en 1998 especializándose en la
distribución mayorista de réplicas a escala de vehículos clásicos. Durante los
primeros cinco años, la empresa consolidó una red de clientes en más de 19 países,
priorizando mercados en Europa y América del Norte. Su estrategia comercial se
basa en ofrecer alta calidad a precios competitivos, lo que le permitió superar
a sus principales competidores en ventas de la categoría 'Classic Cars'. En 2004,
la empresa alcanzó su máximo histórico de ingresos, impulsado por el lanzamiento
de nuevas líneas de productos como motocicletas y camiones a escala.


════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...
⚠️ No puedo resp

---
### 📊 CASO B — Consultas Analíticas y Numéricas sobre el CSV

In [58]:
# ============================================================
# CELDA 10: CASO B.1 — Producto más vendido
# ------------------------------------------------------------
# El agente debe:
#   1. Agrupar el DataFrame por PRODUCTLINE o PRODUCTCODE
#   2. Sumar QUANTITYORDERED
#   3. Identificar el mayor
# ============================================================

respuesta_b1 = preguntar(
    "¿Cuál fue la línea de productos más vendida en términos de cantidad total ordenada?"
    " Muestra las 5 líneas con mayor volumen."
)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: ¿Cuál fue la línea de productos más vendida en términos de cantidad total ordenada? Muestra las 5 líneas con mayor volumen.
════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Agrupar por línea de producto y sumar las cantidades ordenadas\ntop_product_lines = df.groupby('PRODUCTLINE')['QUANTITYORDERED'].sum().sort_values(ascending=False).head(5)\n\ntop_product_lines"}`
responded: Voy a calcular la suma total de `QUANTITYORDERED` para cada línea de productos (`PRODUCTLINE`) y luego ordenar los resultados para mostrar las 5 líneas con mayor volumen.

PRODUCTLINE
Classic Cars        33992
Vintage Cars        21069
Motorcycles         11663
Trucks and Buses    10777
Planes              10727
Name: QUANTITYORDERED, dtype: int64Las 5 líneas de productos con mayor volumen de ventas en términos de **cantidad total ordenada** 

In [59]:
# ============================================================
# CELDA 11: CASO B.2 — Facturación total por año
# ------------------------------------------------------------
# Requiere agrupar por YEAR y sumar la columna SALES.
# ============================================================

respuesta_b2 = preguntar(
    "¿Cuánto se facturó en total en el año 2004? "
    "Muestra también la comparación con 2003 y 2005. "
    "Expresa los valores en dólares con formato $X,XXX,XXX.XX"
)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: ¿Cuánto se facturó en total en el año 2004? Muestra también la comparación con 2003 y 2005. Expresa los valores en dólares con formato $X,XXX,XXX.XX
════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': '# Verifico las ventas totales por año (2003, 2004, 2005)\nventas_por_anio = df.groupby(\'YEAR_ID\')[\'SALES\'].sum().reset_index()\nventas_por_anio = ventas_por_anio[ventas_por_anio[\'YEAR_ID\'].isin([2003, 2004, 2005])]\nventas_por_anio[\'SALES_FORMATTED\'] = ventas_por_anio[\'SALES\'].apply(lambda x: f"${x:,.2f}")\nventas_por_anio'}`


   YEAR_ID       SALES SALES_FORMATTED
0     2003  3516979.54   $3,516,979.54
1     2004  4724162.60   $4,724,162.60
2     2005  1791486.71   $1,791,486.71❌ Error al procesar la pregunta: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","messa

In [60]:
# ============================================================
# CELDA 12: CASO B.3 — Mes con mayores ventas
# ------------------------------------------------------------
# Análisis temporal: ¿en qué mes del año se concentran más ventas?
# ============================================================

respuesta_b3 = preguntar(
    "¿Cuál es el mes del año con mayor facturación total considerando todos los años del dataset? "
    "Muestra los 3 meses con mayor y menor rendimiento."
)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: ¿Cuál es el mes del año con mayor facturación total considerando todos los años del dataset? Muestra los 3 meses con mayor y menor rendimiento.
════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...
❌ Error al procesar la pregunta: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


---
### 🔍 CASO C — Consultas de Filtrado y Segmentación

In [61]:
# ============================================================
# CELDA 13: CASO C.1 — Top 5 clientes con estado 'Shipped'
# ------------------------------------------------------------
# Filtro: STATUS == 'Shipped'
# Agrupación: por CUSTOMERNAME
# Métrica: suma de SALES
# Orden: descendente, top 5
# ============================================================

respuesta_c1 = preguntar(
    "Muestra los 5 clientes con mayores compras totales "
    "que tengan estado de orden 'Shipped'. "
    "Incluye su país y el monto total en dólares."
)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: Muestra los 5 clientes con mayores compras totales que tengan estado de orden 'Shipped'. Incluye su país y el monto total en dólares.
════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...
❌ Error al procesar la pregunta: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


In [62]:
# ============================================================
# CELDA 14: CASO C.2 — Órdenes canceladas por país
# ------------------------------------------------------------
# Filtro: STATUS == 'Cancelled'
# Agrupación: por COUNTRY
# ============================================================

respuesta_c2 = preguntar(
    "¿Cuántas órdenes fueron canceladas (STATUS = 'Cancelled') por país? "
    "Ordena de mayor a menor y muestra solo los países con al menos 1 cancelación."
)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: ¿Cuántas órdenes fueron canceladas (STATUS = 'Cancelled') por país? Ordena de mayor a menor y muestra solo los países con al menos 1 cancelación.
════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...
❌ Error al procesar la pregunta: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


In [63]:
# ============================================================
# CELDA 15: CASO C.3 — Filtrado por rango de precio y deal size
# ------------------------------------------------------------
# Combina filtros numéricos y categóricos
# ============================================================

respuesta_c3 = preguntar(
    "Lista todas las órdenes de tipo 'Large' (DEALSIZE = 'Large') "
    "realizadas en el año 2004 con ventas mayores a $5000. "
    "Muestra: número de orden, cliente, producto, ventas y fecha."
)


════════════════════════════════════════════════════════════
🧑 PREGUNTA: Lista todas las órdenes de tipo 'Large' (DEALSIZE = 'Large') realizadas en el año 2004 con ventas mayores a $5000. Muestra: número de orden, cliente, producto, ventas y fecha.
════════════════════════════════════════════════════════════


> Entering new AgentExecutor chain...
❌ Error al procesar la pregunta: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


---
## 🤖 SECCIÓN 5 — Modo Chat Interactivo (Opcional)

In [64]:
# ============================================================
# CELDA 16: Chat interactivo en consola
# ------------------------------------------------------------
# Permite hacer preguntas libres desde la consola de Colab.
# Escribe 'salir' o 'exit' para terminar el chat.
# ============================================================

def iniciar_chat():
    """
    Inicia una sesión de chat interactivo con el agente.
    Escribe tu pregunta y presiona Enter.
    Escribe 'salir' para terminar.
    """
    print("="*60)
    print("💬 CHAT INTERACTIVO — Agente de Ventas con Mistral AI")
    print("   Escribe tu pregunta o 'salir' para terminar.")
    print("="*60)

    historial = []

    while True:
        try:
            pregunta = input("\n🧑 Tú: ").strip()
        except EOFError:
            break

        if not pregunta:
            print("   (Escribe una pregunta o 'salir')")
            continue

        if pregunta.lower() in ["salir", "exit", "quit"]:
            print("\n👋 ¡Hasta luego! Chat finalizado.")
            break

        respuesta = preguntar(pregunta, mostrar_separador=False)
        print(f"\n🤖 Agente: {respuesta}")
        historial.append({"pregunta": pregunta, "respuesta": respuesta})

    if historial:
        print(f"\n📝 Se realizaron {len(historial)} consulta(s) en esta sesión.")
    return historial


# Descomenta la siguiente línea para iniciar el chat interactivo:
# historial_chat = iniciar_chat()

print("ℹ️  Chat interactivo listo. Descomenta la última línea para usarlo.")

ℹ️  Chat interactivo listo. Descomenta la última línea para usarlo.


---
## 📝 Resumen del Proyecto

| Sección | Componente | Descripción |
|---------|-----------|-------------|
| 1 | Instalación | `langchain`, `langchain-mistralai`, `langchain-experimental` |
| 1 | API Key | `google.colab.userdata` → variable de entorno segura |
| 2 | Normalización | Fechas, números, strings, nulos — pipeline completo |
| 3 | LLM | `ChatMistralAI` — modelo `mistral-large-latest`, temp=0 |
| 3 | Agente | `create_pandas_dataframe_agent` con ReAct loop |
| 4A | Caso A | Comprensión de texto externo con preguntas de lectura |
| 4B | Caso B | Análisis numérico: ventas por año, producto top, mes |
| 4C | Caso C | Filtrado: clientes top, cancelaciones, órdenes grandes |
| 5 | Chat | Modo interactivo opcional en consola |

### 🔄 Ciclo ReAct del Agente

```
Pregunta (lenguaje natural)
         │
         ▼
    [THOUGHT]  ← Mistral razona qué código ejecutar
         │
         ▼
    [ACTION]   ← Ejecuta código Python sobre df
         │
         ▼
 [OBSERVATION] ← Ve el resultado del código
         │
         ▼
[FINAL ANSWER] ← Respuesta en lenguaje natural (español)
```

CREACION DEL RAG

In [65]:
# ============================================================
# CELDA: Construir RAG directamente desde el DataFrame df
# ------------------------------------------------------------
# Cada fila del CSV se convierte en un "documento" de texto
# describiendo esa orden de venta.
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from langchain_core.documents import Document

# ----------------------------------------------------------
# PASO 1: Convertir filas del DataFrame en Documents
# Cada fila se transforma en un texto descriptivo natural
# para que el embedding capture su significado semántico.
# ----------------------------------------------------------
def fila_a_documento(row) -> Document:
    """
    Convierte una fila del DataFrame en un Document de texto.
    El texto describe la orden de forma legible para el embedding.
    """
    texto = (
        f"Orden #{row['ORDERNUMBER']} del {row['ORDERDATE'].strftime('%Y-%m-%d')}. "
        f"Cliente: {row['CUSTOMERNAME']} de {row['CITY']}, {row['COUNTRY']}. "
        f"Producto: {row['PRODUCTLINE']} (código {row['PRODUCTCODE']}). "
        f"Cantidad: {row['QUANTITYORDERED']} unidades a ${row['PRICEEACH']:.2f} c/u. "
        f"Total venta: ${row['SALES']:.2f}. "
        f"Estado: {row['STATUS']}. "
        f"Tamaño del trato: {row['DEALSIZE']}. "
        f"Año: {row['YEAR']}, Mes: {row['MONTH_NAME']}."
    )
    return Document(
        page_content=texto,
        metadata={
            "ordernumber" : row["ORDERNUMBER"],
            "customername": row["CUSTOMERNAME"],
            "country"     : row["COUNTRY"],
            "productline" : row["PRODUCTLINE"],
            "status"      : row["STATUS"],
            "sales"       : row["SALES"],
            "year"        : row["YEAR"],
            "dealsize"    : row["DEALSIZE"],
        }
    )

# Convertir todo el DataFrame
documentos = [fila_a_documento(row) for _, row in df.iterrows()]
print(f"✅ Documentos generados: {len(documentos)} (uno por fila del CSV)")
print(f"\nEjemplo del primer documento:")
print(documentos[0].page_content)

✅ Documentos generados: 2823 (uno por fila del CSV)

Ejemplo del primer documento:
Orden #10107 del 2003-02-24. Cliente: Land of Toys Inc. de NYC, USA. Producto: Motorcycles (código S10_1678). Cantidad: 30 unidades a $95.70 c/u. Total venta: $2871.00. Estado: Shipped. Tamaño del trato: Small. Año: 2003, Mes: February.


In [66]:
# ============================================================
# CELDA: Crear embeddings e indexar en FAISS
# ------------------------------------------------------------
# ADVERTENCIA: Con el dataset completo (~2800 filas) esto
# puede tardar 2-3 minutos y consume tokens de la API.
# Si el rate limit es un problema, usa df.sample(500) primero.
# ============================================================

embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=os.environ["MISTRAL_API_KEY"]
)

# Si tienes rate limit, usa solo una muestra para pruebas:
# documentos_indexar = documentos[:500]
documentos_indexar = documentos  # dataset completo

print(f"⏳ Creando vectorstore con {len(documentos_indexar)} documentos...")
print("   (Esto puede tardar unos minutos)")

vectorstore = FAISS.from_documents(documentos_indexar, embeddings)
retriever   = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # recupera las 5 filas más relevantes
)

print(f"\n✅ FAISS vectorstore listo.")
print(f"   Vectores indexados: {vectorstore.index.ntotal}")

⏳ Creando vectorstore con 2823 documentos...
   (Esto puede tardar unos minutos)

✅ FAISS vectorstore listo.
   Vectores indexados: 2823


In [67]:
# ============================================================
# CELDA: Función RAG sobre el CSV
# ============================================================

from langchain_core.messages import HumanMessage, SystemMessage

def preguntar_rag(pregunta: str) -> str:
    """
    Responde usando SOLO filas recuperadas del CSV via similitud semántica.
    No usa conocimiento externo ni inventa datos.
    """
    print("\n" + "═" * 60)
    print(f"🧑 PREGUNTA RAG: {pregunta}")
    print("═" * 60)

    # Recuperar filas relevantes
    fragmentos = retriever.invoke(pregunta)

    if not fragmentos:
        return "⚠️ No encontré registros relevantes en el dataset."

    print(f"\n📎 Registros recuperados: {len(fragmentos)}")
    for i, f in enumerate(fragmentos):
        print(f"   [{i+1}] {f.page_content[:90]}...")

    # Construir contexto con las filas recuperadas
    contexto = "\n\n".join(
        [f"[Registro {i+1}]:\n{f.page_content}"
         for i, f in enumerate(fragmentos)]
    )

    prompt_sistema = """
    Eres un analista de ventas. Responde ÚNICAMENTE basándote en los registros
    del dataset proporcionados como contexto.
    Si la información no está en los registros recuperados, di:
    "⚠️ Los registros disponibles no contienen suficiente información para responder esto."
    No inventes datos. No uses conocimiento externo. Responde en español.
    """

    prompt_usuario = f"""
    REGISTROS DEL DATASET:
    {contexto}

    PREGUNTA: {pregunta}
    """

    respuesta = llm.invoke([
        SystemMessage(content=prompt_sistema),
        HumanMessage(content=prompt_usuario)
    ])

    print("\n" + "─" * 60)
    print(f"🤖 RESPUESTA RAG:\n{respuesta.content}")
    print("─" * 60)
    return respuesta.content

In [72]:
import time
from langchain_core.messages import HumanMessage, SystemMessage

def preguntar_rag(pregunta: str, max_intentos: int = 3) -> str:
    """
    Responde usando SOLO filas recuperadas del CSV via similitud semántica.
    Incluye reintentos automáticos ante rate limit (429).
    """
    print("\n" + "═" * 60)
    print(f"🧑 PREGUNTA RAG: {pregunta}")
    print("═" * 60)

    fragmentos = retriever.invoke(pregunta)

    if not fragmentos:
        return "⚠️ No encontré registros relevantes en el dataset."

    print(f"\n📎 Registros recuperados: {len(fragmentos)}")
    for i, f in enumerate(fragmentos):
        print(f"   [{i+1}] {f.page_content[:90]}...")

    contexto = "\n\n".join(
        [f"[Registro {i+1}]:\n{f.page_content}"
         for i, f in enumerate(fragmentos)]
    )

    prompt_sistema = """
    Eres un analista de ventas. Responde ÚNICAMENTE basándote en los registros
    del dataset proporcionados como contexto.
    Si la información no está en los registros recuperados, di:
    "⚠️ Los registros disponibles no contienen suficiente información para responder esto."
    No inventes datos. No uses conocimiento externo. Responde en español.
    """

    prompt_usuario = f"""
    REGISTROS DEL DATASET:
    {contexto}

    PREGUNTA: {pregunta}
    """

    for intento in range(max_intentos):
        try:
            respuesta = llm.invoke([
                SystemMessage(content=prompt_sistema),
                HumanMessage(content=prompt_usuario)
            ])
            print("\n" + "─" * 60)
            print(f"🤖 RESPUESTA RAG:\n{respuesta.content}")
            print("─" * 60)
            return respuesta.content

        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                espera = 20 * (intento + 1)
                print(f"⏳ Rate limit. Esperando {espera}s... (intento {intento+1}/{max_intentos})")
                time.sleep(espera)
            else:
                print(f"❌ Error inesperado: {str(e)}")
                return str(e)

    return "❌ No se pudo completar la consulta después de varios intentos."


# ── Pruebas ──────────────────────────────────────────────
preguntar_rag("¿Qué productos compró Euro Shopping Channel?")


════════════════════════════════════════════════════════════
🧑 PREGUNTA RAG: ¿Qué productos compró Euro Shopping Channel?
════════════════════════════════════════════════════════════

📎 Registros recuperados: 5
   [1] Orden #10133 del 2003-06-27. Cliente: Euro Shopping Channel de Madrid, Spain. Producto: Pl...
   [2] Orden #10133 del 2003-06-27. Cliente: Euro Shopping Channel de Madrid, Spain. Producto: Pl...
   [3] Orden #10386 del 2005-03-01. Cliente: Euro Shopping Channel de Madrid, Spain. Producto: Pl...
   [4] Orden #10386 del 2005-03-01. Cliente: Euro Shopping Channel de Madrid, Spain. Producto: Pl...
   [5] Orden #10133 del 2003-06-27. Cliente: Euro Shopping Channel de Madrid, Spain. Producto: Pl...

────────────────────────────────────────────────────────────
🤖 RESPUESTA RAG:
Euro Shopping Channel compró los siguientes productos según los registros disponibles:

1. **Planes (código S24_4278)**
2. **Planes (código S24_1785)**
3. **Planes (código S700_1691)**
4. **Planes (código

'Euro Shopping Channel compró los siguientes productos según los registros disponibles:\n\n1. **Planes (código S24_4278)**\n2. **Planes (código S24_1785)**\n3. **Planes (código S700_1691)**\n4. **Planes (código S24_2841)**'